# Validare un modello



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


Hai costruito un modello. Ma quanto è valido? La **validazione** misura la qualità e permette di migliorare il modello attraverso esperimenti successivi.

# Che cos'è la validazione?
Quasi ogni modello va valutato. In molte applicazioni conta quanto le previsioni si avvicinano ai valori reali. Un errore frequente è confrontare previsioni e target sugli stessi dati usati per addestrare il modello. Tra poco vedremo perché è problematico.

Prima serve una misura sintetica: confrontare a mano 10.000 prezzi previsti e reali sarebbe poco utile. Useremo il **Mean Absolute Error**, **MAE**, cioè l'errore assoluto medio. Per ogni casa:

```text
error = actual - predicted
```

Se il prezzo reale è 150.000 dollari e quello previsto è 100.000, l'errore è 50.000. Prendiamo il valore assoluto di ogni errore e poi la media. Il MAE si interpreta così:

> Le nostre previsioni si discostano in media di circa X dal valore reale.

Prima di calcolarlo ricostruiamo il modello nel codice seguente, nascosto nell'interfaccia originale di Kaggle.


In [ ]:
# Codice di caricamento dei dati nascosto.
import pandas as pd

# Carica i dati.
melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
# Escludi le righe con prezzo mancante.
filtered_melbourne_data = melbourne_data.dropna(axis=0)
# Scegli il target e le feature.
y = filtered_melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 
                        'YearBuilt', 'Lattitude', 'Longtitude']
X = filtered_melbourne_data[melbourne_features]

from sklearn.tree import DecisionTreeRegressor
# Definisci il modello.
melbourne_model = DecisionTreeRegressor()
# Addestra il modello.
melbourne_model.fit(X, y)

Con il modello pronto, calcoliamo l'errore assoluto medio:


In [ ]:
from sklearn.metrics import mean_absolute_error

predicted_home_prices = melbourne_model.predict(X)
mean_absolute_error(y, predicted_home_prices)

# Il problema dei risultati sui dati di training
Il punteggio appena calcolato è **in-sample**: le stesse case servono per addestramento e valutazione.

Immagina che nel mercato immobiliare il colore della porta non abbia alcun rapporto con il prezzo. Per caso, però, nel nostro campione tutte le case con una porta verde sono costose. Il modello individua questa associazione e assegna prezzi elevati alle case con porte verdi. Sui dati di training sembra accurato, ma su case nuove potrebbe sbagliare molto se l'associazione non si ripete.

Il valore pratico di un modello dipende dalle previsioni su dati nuovi. Per misurarlo escludiamo una parte dei dati dall'addestramento e la usiamo per la valutazione. È il **validation set**, l'insieme di validazione.

# Il codice
La funzione `train_test_split` di scikit-learn divide i dati. Una parte serve per addestrare il modello, l'altra per calcolare `mean_absolute_error` su esempi mai visti durante il training:


In [ ]:
from sklearn.model_selection import train_test_split

# Dividi feature e target nei set di training e validazione.
# La suddivisione usa un generatore di numeri casuali. Assegnando un numero
# all'argomento random_state si ottiene la stessa suddivisione ogni volta che si
# eseguire nuovamente lo script.
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)
# Definisci il modello.
melbourne_model = DecisionTreeRegressor()
# Addestra il modello.
melbourne_model.fit(train_X, train_y)

# Prevedi i prezzi sui dati di validazione.
val_predictions = melbourne_model.predict(val_X)
print(mean_absolute_error(val_y, val_predictions))

# Una differenza notevole
Nell'esempio originale il MAE passa da circa 500 dollari sui dati di training a oltre 250.000 sui dati nuovi. È la differenza tra un modello apparentemente quasi perfetto e uno poco utile nella pratica. Il prezzo medio nel validation set è circa 1,1 milioni: l'errore è circa un quarto del prezzo medio.

Possiamo migliorare il modello provando feature e algoritmi diversi. I risultati numerici dipendono dal dataset e dalla versione delle librerie.


Prima di migliorare il modello, svolgi l'**[esercizio sulla validazione](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex4.html)**.
